# Post-Hoc Evaluation of Trained Model
This notebook performs post-hoc evaluation of a trained model using W&B. It includes validation dataset evaluation and plot generation.

## Setup and Imports

In [1]:
# Import required libraries
%load_ext autoreload
%autoreload 1
import torch
import wandb
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from src.config import pretty_config, find_wandb_run, get_current_config
from src.evaluation import PlotsCallback
import src.models
import src.data_loaders

# Set up logging
import logging
from src.logging_util import handler

log_level = logging.INFO
logging.basicConfig(level=log_level, handlers=[handler])
logging.getLogger('src').setLevel(log_level)
logger = logging.getLogger(__name__)
logger.setLevel(log_level)

In [2]:
PROJECT = "flowtoy"
FIND_RUN = "Prof2_art_delete_e10_levy"
selected_run = find_wandb_run(FIND_RUN)

 ✅ Run ID: 1hz07g5h, Run Name: Prof2_art_delete_e10_levy
 Created at: 2025-04-21T17:02:55Z
   Run URL: https://wandb.ai/tresoor/flowtoy/runs/1hz07g5h


In [3]:
# Load the configuration file
config = selected_run.config
print(pretty_config(config))

{'Attn': [True, False, False, False],
 'Cols': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'],
 'Crop': 2000,
 'data': {'dir': './data/',
          'cols': {'c': ['IP', 'gas_fringes', 'NBI', 'ECRH', 'a_minor', 'KAPPA', 'DELTA'],
                   'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'],
                   'meta': ['ShotNum', 'time'],
                   'label': 'LHD_label'},
          'file': '2024_05_01-NaNsFiltered.parquet',
          'Class': 'ShotFlowDS',
          'seq_length': 256,
          'test_shots': [53623, 57013, 57094, 57732, 60813, 60814, 61028, 61237, 63306, 63878, 64365, 64386, 64393, 64678, 64686, 64770, 64857, 65481, 67112,
                         68631, 68697, 69514, 73368, 73631, 73935, 75264, 76304, 76702, 77089, 77193, 77196, 77409, 77595, 77598, 77599, 77602, 77604],
          'crop_margin': 2000,
          'pre_shuffle': True,
          'sample_rate': 10000,
          'train_shots': [26386, 29511, 30043, 30197, 30225, 30262, 30268, 30290, 30310, 31211, 

## Load Model and Configuration from W&B
Use the W&B API to load the trained model and its configuration. Initialize the model using the configuration parameters.

In [4]:
# Load configuration and initialize W&B
from src.models import FlowModule
eval_run = wandb.init(
    project=PROJECT,
    name=selected_run.name,
    id=selected_run.id,
    resume="must",
    mode="disabled",
    config=config,
)

# Load model configuration
C = get_current_config()
ModelClass = getattr(src.models, C.model.Class)
assert issubclass(ModelClass, FlowModule), "ModelClass must be a subclass of FlowModule"

model = ModelClass(**C.model.params)

# Log model summary
logger.info("Model loaded. Summary:")
model.log_summary(C)
C

10:16:15 src.models.flow[INFO]:Will do 1 matches per batch, stepping every 1, so 1 steps per batch.
10:16:15 (+ 0.00s) __main__[INFO]:Model loaded. Summary:
10:16:29 (+13.14s) src.models.flow[INFO]:Model expected input shape: [(128, 5, 256), (128,), {'x_history': (128, 5, 256), 'position_sequence': (128, 512), 'c': (128, 7, 512)}]


Layer (type (var_name):depth-idx)                       Input Shape               Output Shape              Kernel Shape              Param #                   Mult-Adds
ConditionalUNet (ConditionalUNet)                       [128, 5, 256]             [128, 5, 256]             --                        --                        --
├─TimeEmbedding (pos_emb): 1-1                          [128, 512]                [128, 512, 4]             --                        --                        --
│    └─Linear (lin1): 2-1                               [65536, 32]               [65536, 32]               --                        1,056                     69,206,016
│    └─SiLU (act): 2-2                                  [65536, 32]               [65536, 32]               --                        --                        --
│    └─Linear (lin2): 2-3                               [65536, 32]               [65536, 4]                --                        132                       8,650,752


{'Attn': [True, False, False, False], 'Cols': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'], 'Crop': 2000, 'data': {'dir': './data/', 'cols': {'c': ['IP', 'gas_fringes', 'NBI', 'ECRH', 'a_minor', 'KAPPA', 'DELTA'], 'x': ['FIR_core', 'PD', 'DML', 'POHM', 'Z_axis'], 'meta': ['ShotNum', 'time'], 'label': 'LHD_label'}, 'file': '2024_05_01-NaNsFiltered.parquet', 'Class': 'ShotFlowDS', 'seq_length': 256, 'test_shots': [53623, 57013, 57094, 57732, 60813, 60814, 61028, 61237, 63306, 63878, 64365, 64386, 64393, 64678, 64686, 64770, 64857, 65481, 67112, 68631, 68697, 69514, 73368, 73631, 73935, 75264, 76304, 76702, 77089, 77193, 77196, 77409, 77595, 77598, 77599, 77602, 77604], 'crop_margin': 2000, 'pre_shuffle': True, 'sample_rate': 10000, 'train_shots': [26386, 29511, 30043, 30197, 30225, 30262, 30268, 30290, 30310, 31211, 31554, 31650, 31718, 31807, 31839, 32191, 32195, 32592, 32716, 32911, 33188, 33271, 33281, 33459, 33567, 33942, 34010, 34309, 42197, 42514, 43454, 45103, 45105, 46853, 47962, 

## Load Validation Dataset
Load the validation dataset using the same DataSetClass and parameters as in `run.py`.

In [5]:
# Load validation dataset
DataSetClass = getattr(src.data_loaders, C.data.Class)
val_set = DataSetClass(**C.data, train=False)

# Create DataLoader for validation dataset
val_loader = DataLoader(
    val_set,
    batch_size=1,
    shuffle=False,
)

logger.info("Validation dataset loaded.")

10:16:29 (+ 0.62s) src.data_loaders[INFO]:Using 37 shots from ./data/2024_05_01-NaNsFiltered.parquet
10:16:29 (+ 0.18s) src.data_loaders[INFO]:Column 'time_step      ': min=0.0000      max=26138.0000  mean=8040.4892   std=5206.8242   nans=0         
10:16:29 (+ 0.00s) src.data_loaders[INFO]:Column 'ShotNum        ': min=53623.0000  max=77604.0000  mean=69145.2513  std=7638.5923   nans=0         
10:16:29 (+ 0.01s) src.data_loaders[INFO]:Column 'IP             ': min=0.0000      max=1.0000      mean=0.4399      std=0.1395      nans=0         
10:16:29 (+ 0.01s) src.data_loaders[INFO]:Column 'gas_fringes    ': min=0.0000      max=1.0000      mean=0.3031      std=0.1447      nans=0         
10:16:29 (+ 0.01s) src.data_loaders[INFO]:Column 'NBI            ': min=0.0000      max=1.0000      mean=0.3054      std=0.3801      nans=0         
10:16:29 (+ 0.01s) src.data_loaders[INFO]:Column 'ECRH           ': min=0.0000      max=1.0000      mean=0.0649      std=0.1680      nans=0         
10:16

## Run Evaluation and Generate Plots
Run the evaluation functions on the validation dataset and generate plots using the plot functions defined in `evaluation.py`.

In [7]:
# Run evaluation
# N_STEPS = C.evaluation.n_steps
N_STEPS = 50
logger.info("Starting evaluation using %s steps...", N_STEPS)
batch = next(iter(val_loader))
evaluation_outputs = model.evaluate(batch, n_steps=N_STEPS)
logger.info("Evaluation completed.")
evaluation_outputs


10:19:11 (+158.37s) __main__[INFO]:Starting evaluation using 50 steps...
Integrating path: 100%|██████████| 49/49 [00:09<00:00,  5.19it/s]
10:19:20 (+ 9.66s) __main__[INFO]:Evaluation completed.


{'meta': {'shot_number': tensor([77193]),
  'start': tensor([1.5134], dtype=torch.float64),
  'end': tensor([1.5390], dtype=torch.float64),
  'history_start': tensor([1.4878], dtype=torch.float64)},
 'conditioning_input': {'x_history': tensor([[[0.5676, 0.5657, 0.5639,  ..., 0.5909, 0.5857, 0.5900],
           [0.0696, 0.0724, 0.0734,  ..., 0.0778, 0.1376, 0.1437],
           [0.5427, 0.5467, 0.5465,  ..., 0.5471, 0.5462, 0.5444],
           [0.1323, 0.1323, 0.1323,  ..., 0.1280, 0.1280, 0.1280],
           [0.2067, 0.2067, 0.2067,  ..., 0.2114, 0.2114, 0.2114]]]),
  'position_sequence': tensor([[1.4878, 1.4879, 1.4880, 1.4881, 1.4882, 1.4883, 1.4884, 1.4885, 1.4886,
           1.4887, 1.4888, 1.4889, 1.4890, 1.4891, 1.4892, 1.4893, 1.4894, 1.4895,
           1.4896, 1.4897, 1.4898, 1.4899, 1.4900, 1.4901, 1.4902, 1.4903, 1.4904,
           1.4905, 1.4906, 1.4907, 1.4908, 1.4909, 1.4910, 1.4911, 1.4912, 1.4913,
           1.4914, 1.4915, 1.4916, 1.4917, 1.4918, 1.4919, 1.4920, 1.4921, 

In [ ]:
evaluation_outputs

In [ ]:

# Generate plots
logger.info("Generating plots...")
plots_callback = PlotsCallback(C.evaluation)
    plots_callback.call_plot_functions(
        evaluation_output=output,
        trainval="val",
        global_step=0,
        title_base="Post-Hoc Evaluation"
    )

logger.info("Evaluation and plot generation complete.")